# 영상 → 점 · 카메라 위치 (MapAnything, Colab T4)

메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 고른 뒤, 위에서부터 셀을 하나씩 실행하세요.

결과로 `points.npz`(점·색·카메라)와 `frames.zip`(추출한 사진 24장)이 내려옵니다.

## 1. GPU 확인

In [ ]:
import torch, time, os, json
T0 = time.time()
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸고 다시 실행하세요."
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability()
GPU_NAME = p.name
print(f"{p.name} / 메모리 {p.total_memory/1e9:.1f}GB / 계산 능력 {cap[0]}.{cap[1]}")

## 2. 설치 (약 1분)

In [ ]:
%%capture
!git clone -q --depth 1 https://github.com/facebookresearch/map-anything.git
!cd map-anything && pip install -q -e .

In [ ]:
import sys
sys.path.insert(0, '/content/map-anything')   # 설치 직후 import 실패 방지
import mapanything
print("mapanything 준비됨", f"{time.time()-T0:.0f}초")

## 3. 모델 미리 받기 (약 3분, 4.91GB)

영상 업로드 전에 받아 두면 기다리는 시간이 겹칩니다.

In [ ]:
from mapanything.models import MapAnything
t = time.time()
model = MapAnything.from_pretrained("facebook/map-anything-apache").to("cuda").eval()
print(f"모델 준비 {time.time()-t:.0f}초")

## 4. 영상 올리기

`345.mp4` 를 선택하세요.

In [ ]:
from google.colab import files
up = files.upload()
VIDEO = os.path.join('/content', list(up.keys())[0])
print("올린 파일:", VIDEO, f"{os.path.getsize(VIDEO)/1e6:.1f}MB")

## 5. 프레임 24장 고르게 뽑기

In [ ]:
import subprocess, glob
dur = float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',VIDEO]).decode().strip())
info = subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width,height','-of','csv=p=0',VIDEO]).decode().strip()
print("영상 길이", round(dur,1), "초 / 해상도", info)
N_FRAMES = 24
fps = N_FRAMES / dur
os.makedirs('/content/frames', exist_ok=True)
for f in glob.glob('/content/frames/*.jpg'): os.remove(f)
# 긴 변을 1280으로 줄여 저장 (모델이 어차피 더 작게 줄여 씀)
subprocess.run(['ffmpeg','-v','error','-i',VIDEO,'-vf',f"fps={fps},scale='if(gt(iw,ih),1280,-2)':'if(gt(iw,ih),-2,1280)'",'-q:v','2','/content/frames/f_%03d.jpg'],check=True)
paths = sorted(glob.glob('/content/frames/*.jpg'))[:N_FRAMES]
print(len(paths), "장 추출")
import matplotlib.pyplot as plt
from PIL import Image
fig, ax = plt.subplots(3, 8, figsize=(20, 7))
for a, pth in zip(ax.ravel(), paths):
    a.imshow(Image.open(pth)); a.axis('off'); a.set_title(os.path.basename(pth), fontsize=7)
plt.tight_layout(); plt.show()

번지거나 흔들린 장이 많은지 눈으로 확인하세요.

## 6. 복원 (T4는 반드시 fp16)

In [ ]:
from mapanything.utils.image import load_images
views = load_images(paths)
IN_SIZE = tuple(views[0]['img'].shape[-2:])
print("모델 입력 크기", IN_SIZE)
torch.cuda.reset_peak_memory_stats(); t = time.time()
pred = model.infer(views, memory_efficient_inference=True, minibatch_size=1,
                   use_amp=True, amp_dtype="fp16",
                   apply_mask=True, mask_edges=True, apply_confidence_mask=False)
torch.cuda.synchronize()
RECON_SEC = time.time() - t
PEAK_GB = torch.cuda.max_memory_allocated()/1e9
print(f"복원 {RECON_SEC:.1f}초, 최대 GPU 메모리 {PEAK_GB:.2f}GB")

## 7. 멀리 튄 점 버리고 카메라 퍼짐 비 계산

In [ ]:
import numpy as np
pts, cols, cams = [], [], []
for v in pred:
    m = v['mask'][0, ..., 0].cpu().numpy()
    pts.append(v['pts3d'][0].cpu().numpy()[m])
    cols.append((v['img_no_norm'][0].cpu().numpy()*255).clip(0,255).astype(np.uint8)[m])
    cams.append(v['camera_poses'][0].cpu().numpy())
xyz = np.concatenate(pts); rgb = np.concatenate(cols); cams = np.stack(cams)
N_RAW = len(xyz)
lo = np.percentile(xyz, 0.5, axis=0); hi = np.percentile(xyz, 99.5, axis=0)
keep = np.all((xyz > lo) & (xyz < hi), axis=1)
xyz, rgb = xyz[keep], rgb[keep]
cpos = cams[:, :3, 3]
cam_diag = float(np.linalg.norm(cpos.max(0)-cpos.min(0)))
scene_diag = float(np.linalg.norm(xyz.max(0)-xyz.min(0)))
RATIO = cam_diag/scene_diag
print(f"점 {N_RAW:,}개 -> 이상치 제거 후 {len(xyz):,}개")
print(f"카메라 상자 대각선 {cam_diag:.3f} / 장면 상자 대각선 {scene_diag:.3f} = 퍼짐 비 {RATIO:.4f}")
print("걸어 다니며 찍은 것으로 보입니다" if RATIO>0.1 else "카메라가 거의 안 움직였습니다. 촬영을 다시 볼 필요가 있습니다")

카메라 위치(점)와 바라보는 방향(화살표)을 장면과 함께 봅니다. 실제로 걸은 경로와 비슷한지 확인하세요.

In [ ]:
sub = np.random.default_rng(0).choice(len(xyz), min(40000, len(xyz)), replace=False)
fwd = cams[:, :3, 2]   # 카메라가 바라보는 방향
fig = plt.figure(figsize=(14,6))
for k,(el,az) in enumerate([(90,-90),(20,-60)]):
    ax = fig.add_subplot(1,2,k+1,projection='3d')
    ax.scatter(xyz[sub,0],xyz[sub,1],xyz[sub,2],c=rgb[sub]/255.0,s=0.3)
    ax.scatter(cpos[:,0],cpos[:,1],cpos[:,2],c='red',s=25)
    L = scene_diag*0.08
    ax.quiver(cpos[:,0],cpos[:,1],cpos[:,2],fwd[:,0]*L,fwd[:,1]*L,fwd[:,2]*L,color='red')
    ax.view_init(elev=el,azim=az); ax.set_title("위에서" if k==0 else "비스듬히")
plt.tight_layout(); plt.show()

## 8. 저장하고 내려받기

코랩 세션은 유휴 시 자동 종료되므로 **반드시 내려받으세요.**

In [ ]:
np.savez_compressed('/content/points.npz', xyz=xyz.astype(np.float32), rgb=rgb, cams=cams.astype(np.float32))
meta = dict(gpu=GPU_NAME, recon_sec=round(RECON_SEC,1), peak_gpu_gb=round(PEAK_GB,2),
            n_points_raw=int(N_RAW), n_points=int(len(xyz)), cam_diag=cam_diag, scene_diag=scene_diag,
            spread_ratio=RATIO, n_frames=len(paths), model_input_size=list(IN_SIZE),
            video_duration_sec=dur, video_info=info)
json.dump(meta, open('/content/meta.json','w'), indent=2, ensure_ascii=False)
!cd /content && zip -q -r frames.zip frames
print(json.dumps(meta, indent=2, ensure_ascii=False))
print("points.npz", f"{os.path.getsize('/content/points.npz')/1e6:.1f}MB", "/ frames.zip", f"{os.path.getsize('/content/frames.zip')/1e6:.1f}MB")
files.download('/content/points.npz'); files.download('/content/meta.json'); files.download('/content/frames.zip')
print(f"총 {time.time()-T0:.0f}초")

끝나면 **런타임 → 세션 연결 해제 및 삭제** 로 GPU를 반납하세요.